# Alpaca market data

Pull recent 15-minute SPY bars and look at price and trading volume.

Install the packages once:

```bash
python -m pip install alpaca-py pandas plotly python-dotenv ipykernel
```

Create a `.env` file in the same folder as this notebook:

```text
API_KEY=your_alpaca_key
SECRET_KEY=your_alpaca_secret
```

In [6]:
import os
from datetime import datetime, timedelta, timezone

import pandas as pd
import plotly.graph_objects as go
from alpaca.data.enums import DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from dotenv import load_dotenv

load_dotenv(".env")

API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")
if not API_KEY or not SECRET_KEY:
    raise ValueError("Add API_KEY and SECRET_KEY to the .env file beside this notebook")

SYMBOL = "SPY"
BAR_MINUTES = 15
DAYS_TO_PULL = 14

## Pull the data

In [7]:
end = datetime.now(timezone.utc)
start = end - timedelta(days=DAYS_TO_PULL)

client = StockHistoricalDataClient(API_KEY, SECRET_KEY)
request = StockBarsRequest(
    symbol_or_symbols=SYMBOL,
    timeframe=TimeFrame(BAR_MINUTES, TimeFrameUnit.Minute),
    start=start,
    end=end,
    feed=DataFeed.IEX,
)

bars = client.get_stock_bars(request).df.reset_index()
if bars.empty:
    raise ValueError(f"Alpaca returned no bars for {SYMBOL}")

bars["timestamp"] = pd.to_datetime(bars["timestamp"], utc=True)
bars = bars.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

new_york_time = bars["timestamp"].dt.tz_convert("America/New_York")
session_date = new_york_time.dt.date
recent_sessions = sorted(session_date.unique())[-5:]
chart_data = bars.loc[session_date.isin(recent_sessions)].copy()
chart_time = chart_data["timestamp"].dt.tz_convert("America/New_York")

print(f"Pulled {len(bars):,} {SYMBOL} bars")
bars.tail()

Pulled 262 SPY bars


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
257,SPY,2026-09-15 14:30:00+00:00,757.480,757.620,756.885,757.200,61804.0,1017.0,757.295753
258,SPY,2026-09-15 14:45:00+00:00,757.200,757.370,756.180,756.430,76889.0,1100.0,756.720995
259,SPY,2026-09-15 15:00:00+00:00,756.440,757.255,756.440,757.015,60047.0,832.0,756.925776
260,SPY,2026-09-15 15:15:00+00:00,756.965,757.495,756.810,757.330,38225.0,575.0,757.250311
261,SPY,2026-09-15 15:30:00+00:00,757.480,757.555,756.945,757.420,33422.0,498.0,757.223503


## Price

In [8]:
price_chart = go.Figure(go.Candlestick(
    x=chart_time,
    open=chart_data["open"],
    high=chart_data["high"],
    low=chart_data["low"],
    close=chart_data["close"],
    name=SYMBOL,
))
price_chart.update_layout(
    title=f"{SYMBOL} — 15-minute candles",
    xaxis_title="New York time",
    yaxis_title="Price (USD)",
    xaxis_rangeslider_visible=False,
    height=600,
)
price_chart.show()

## Volume

In [9]:
volume_colors = [
    "#16a34a" if close >= open_ else "#dc2626"
    for open_, close in zip(chart_data["open"], chart_data["close"])
]

volume_chart = go.Figure(go.Bar(
    x=chart_time,
    y=chart_data["volume"],
    marker_color=volume_colors,
    name="Volume",
))
volume_chart.update_layout(
    title=f"{SYMBOL} — 15-minute volume",
    xaxis_title="New York time",
    yaxis_title="Shares",
    height=400,
    showlegend=False,
)
volume_chart.show()